============================================================
SPARK DEEP INTERNALS - Lead Engineer Interview
============================================================
Topic: Cluster Architecture, Resource Management, DAG, Stages
============================================================

# Q1: Spark Cluster Architecture (YARN)


┌─────────────────────────────────────────────────────────────────┐
│                    CLUSTER MANAGER (YARN)                        │
│  ┌─────────────────────────────────────────────────────────────┐ │
│  │              RESOURCE MANAGER (Master)                       │ │
│  │  - Accepts applications                                      │ │
│  │  - Allocates cluster resources                               │ │
│  │  - Monitors NodeManagers                                     │ │
│  └─────────────────────────────────────────────────────────────┘ │
└─────────────────────────────────────────────────────────────────┘
                              │
        ┌─────────────────────┼─────────────────────┐
        ▼                     ▼                     ▼
┌──────────────────┐  ┌──────────────────┐  ┌──────────────────┐
│   NODE MANAGER    │  │   NODE MANAGER    │  │   NODE MANAGER    │
│  ┌─────────────┐  │  │  ┌─────────────┐  │  │  ┌─────────────┐  │
│  │ APPLICATION │  │  │  │  EXECUTOR   │  │  │  │  EXECUTOR   │  │
│  │   MASTER    │  │  │  │ ┌─────────┐ │  │  │  │ ┌─────────┐ │  │
│  │  (Driver)   │  │  │  │ │  Task   │ │  │  │  │ │  Task   │ │  │
│  │             │  │  │  │ │  Task   │ │  │  │  │ │  Task   │ │  │
│  └─────────────┘  │  │  │ └─────────┘ │  │  │  │ └─────────┘ │  │
│                   │  │  └─────────────┘  │  │  └─────────────┘  │
└──────────────────┘  └──────────────────┘  └──────────────────┘

COMPONENTS:
1. RESOURCE MANAGER: Cluster-wide resource allocation
2. NODE MANAGER: Per-node agent, manages containers
3. APPLICATION MASTER: Per-application, negotiates resources
4. CONTAINER: Allocated resources (CPU, memory)
5. EXECUTOR: JVM process running tasks

FLOW:
1. Client submits spark-submit
2. Resource Manager allocates container for Application Master
3. Application Master requests containers for Executors
4. Node Managers launch Executors
5. Driver (in AM) schedules tasks to Executors


# Q2: spark-submit Flow


WHAT HAPPENS WHEN YOU RUN spark-submit:

1. CLIENT SUBMISSION
   spark-submit --master yarn --deploy-mode cluster my_job.py
   
2. RESOURCE MANAGER RECEIVES REQUEST
   - Validates application
   - Allocates container for Application Master
   
3. APPLICATION MASTER STARTS
   - In cluster mode: Driver runs inside AM
   - In client mode: Driver runs on client machine
   - AM requests Executor containers from RM
   
4. EXECUTORS LAUNCH
   - Node Managers start Executor JVMs
   - Executors register with Driver
   
5. DRIVER CREATES SparkContext
   - Parses your code
   - Builds DAG of stages
   
6. JOB EXECUTION
   - Driver sends tasks to Executors
   - Executors process data, return results
   
7. COMPLETION
   - Results collected at Driver
   - Executors terminated
   - AM reports completion to RM

CODE TO DAG FLOW:
df.read() → filter() → groupBy() → count() → show()
    │           │           │          │        │
    │           │           │          │        └── ACTION (triggers execution)
    │           │           │          └── Transformation (wide)
    │           │           └── Transformation (wide - shuffle boundary)
    │           └── Transformation (narrow)
    └── Transformation (narrow)

STAGES CREATED:
Stage 0: read + filter (narrow, pipelined)
--- SHUFFLE ---
Stage 1: groupBy partial aggregation
--- SHUFFLE ---
Stage 2: final aggregation + count + show


# Q3: DAG, Jobs, Stages, and Tasks


HIERARCHY:
Application → Jobs → Stages → Tasks

APPLICATION: One SparkContext, one application
JOB: Triggered by each ACTION (count, show, write)
STAGE: Separated by shuffle boundaries (wide transformations)
TASK: One task per partition per stage

EXAMPLE:
df = spark.read.parquet("data")     # Stage 0: read
    .filter(col("x") > 10)           # Stage 0: filter (narrow)
    .groupBy("key")                  # --- SHUFFLE BOUNDARY ---
    .agg(sum("value"))               # Stage 1: aggregate
    .orderBy("key")                  # --- SHUFFLE BOUNDARY ---
    .show()                          # Stage 2: collect (ACTION)

If data has 100 partitions:
- Stage 0: 100 tasks (one per partition)
- Stage 1: 200 tasks (default shuffle partitions)
- Stage 2: 200 tasks

TOTAL: 500 tasks for this job


# Q4: SparkContext vs SparkSession


SparkContext (SC):
- Entry point for Spark core (RDD API)
- Creates one per JVM
- Manages cluster connection
- Creates RDDs

SparkSession:
- Entry point for DataFrame/Dataset API (Spark 2.0+)
- Wraps SparkContext internally
- Unified entry point for SQL, Hive, Streaming
- Recommended for all modern Spark code

CODE:
# Old way (Spark 1.x)
from pyspark import SparkContext, SparkConf
conf = SparkConf().setAppName("MyApp")
sc = SparkContext(conf=conf)

# Modern way (Spark 2.0+)
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("MyApp").getOrCreate()
sc = spark.sparkContext  # Access underlying SC if needed


# Q5: RDD Internals


RDD (Resilient Distributed Dataset):
- Immutable, partitioned collection
- Lineage graph for fault tolerance
- Lazy evaluation

RDD PROPERTIES:
1. Partitions: List of data chunks
2. Dependencies: Parent RDDs (narrow/wide)
3. Compute function: How to compute each partition
4. Partitioner: Optional, for key-value RDDs
5. Preferred locations: Data locality hints

LINEAGE vs CHECKPOINTING:
- Lineage: Re-compute from source if partition lost
- Checkpoint: Save to HDFS, break lineage (faster recovery)

WHEN TO USE RDD:
1. Low-level transformations (mapPartitions)
2. Unstructured data
3. Fine-grained control
4. Legacy code

WHEN TO USE DATAFRAME:
1. Structured data (99% of cases)
2. Need Catalyst optimization
3. SQL queries
4. Interoperability with other tools


# Q6: Narrow vs Wide Dependencies (Deep Dive)


NARROW DEPENDENCY:
- Each parent partition → ONE child partition
- No data movement across nodes
- Can be pipelined in single task
- Examples: map, filter, flatMap, union

WIDE DEPENDENCY:
- Each parent partition → MULTIPLE child partitions
- Requires SHUFFLE (data movement)
- Stage boundary
- Examples: groupByKey, reduceByKey, join, repartition

WHY THIS MATTERS:
1. Narrow: Fault recovery only needs one parent partition
2. Wide: Fault recovery may need ALL parent partitions
3. Wide: Creates shuffle files on disk (expensive)

SHUFFLE INTERNALS:
┌─────────────┐     Map Side        ┌─────────────┐
│ Partition 0 │ ──Write────────────►│ Shuffle     │
│ Partition 1 │ ──Write────────────►│ Files       │
│ Partition 2 │ ──Write────────────►│ (Local Disk)│
└─────────────┘                     └──────┬──────┘
                                           │
                                     Sort &│Merge
                                           │
         ┌─────────────────────────────────┼─────────────────────┐
         ▼                                 ▼                     ▼
    ┌─────────┐                      ┌─────────┐            ┌─────────┐
    │ Part A  │                      │ Part B  │            │ Part C  │
    │(all k=A)│                      │(all k=B)│            │(all k=C)│
    └─────────┘                      └─────────┘            └─────────┘


# Q7: Catalyst Optimizer Phases


CATALYST OPTIMIZATION PHASES:

1. ANALYSIS
   - Parse SQL/DataFrame to unresolved logical plan
   - Resolve table/column names using Catalog
   - Type checking
   
2. LOGICAL OPTIMIZATION
   - Predicate pushdown (filter early)
   - Projection pruning (select only needed columns)
   - Constant folding (evaluate 1+1 at compile time)
   - Boolean simplification
   - Null propagation
   
3. PHYSICAL PLANNING
   - Generate multiple physical plans
   - Cost-based optimization (CBO) selects best
   - Choose join strategies (broadcast vs sort-merge)
   
4. CODE GENERATION
   - Whole-stage code generation
   - Generate Java bytecode
   - Avoids virtual function calls

SEE THE PLANS:
df.explain()          # Physical plan only
df.explain(True)      # All plans
df.explain("cost")    # With cost estimates
df.explain("formatted")  # Human-readable

CBO (Cost-Based Optimization):
- Uses table statistics (row count, column stats)
- Estimates cost of each plan
- Requires: spark.sql.cbo.enabled = true
- Run: ANALYZE TABLE table COMPUTE STATISTICS


# Q8: Data Locality Levels


Spark tries to schedule tasks close to data:

LOCALITY LEVELS (best to worst):
1. PROCESS_LOCAL: Data in same JVM (cached)
2. NODE_LOCAL: Data on same node (different executor)
3. NO_PREF: No preference (e.g., generated data)
4. RACK_LOCAL: Data on same rack
5. ANY: Data on any node (worst)

CONFIGURATION:
spark.locality.wait = 3s (default)
spark.locality.wait.process = 3s
spark.locality.wait.node = 3s
spark.locality.wait.rack = 3s

Spark waits up to spark.locality.wait before falling back.


# Q9: Speculative Execution


PROBLEM: One slow task holds up entire stage (straggler)

SOLUTION: Launch duplicate task on another executor

CONFIGURATION:
spark.speculation = true (default false)
spark.speculation.interval = 100ms
spark.speculation.multiplier = 1.5
spark.speculation.quantile = 0.75

HOW IT WORKS:
1. 75% of tasks complete
2. If remaining task is 1.5x slower than median, launch speculative copy
3. First to finish wins, other killed

CAUTION:
- Don't use with non-idempotent operations
- Don't use if tasks have side effects (writes)


# Q10: Fault Tolerance in Spark


HOW SPARK HANDLES FAILURES:

TASK FAILURE:
- Retry task on same/different executor
- Default: 4 retries (spark.task.maxFailures)
- If all retries fail, stage fails

STAGE FAILURE:
- Retry entire stage
- Default: 4 retries (spark.stage.maxConsecutiveAttempts)
- Uses shuffle files if available

EXECUTOR FAILURE:
- Task redistributed to other executors
- Cached data lost (re-computed from lineage)

DRIVER FAILURE:
- Application fails
- For resilience: Use checkpointing, cluster mode

RDD LINEAGE:
- Each RDD knows how to recompute from parents
- Narrow: Only need one parent partition
- Wide: Need all parent partitions (or shuffle files)


In [ ]:
if __name__ == "__main__":
    from pyspark.sql import SparkSession
    
    spark = SparkSession.builder \
        .appName("InternalsDemo") \
        .master("local[*]") \
        .getOrCreate()
    
    # Access SparkContext
    sc = spark.sparkContext
    
    print(f"Application ID: {sc.applicationId}")
    print(f"Default Parallelism: {sc.defaultParallelism}")
    print(f"Master: {sc.master}")
    
    # Create sample data and show DAG
    df = spark.range(0, 1000) \
        .withColumn("key", (df["id"] % 10).cast("string")) \
        .groupBy("key").count()
    
    print("\n--- Physical Plan (shows stages) ---")
    df.explain()
    
    print("\n--- Full Plan (Logical + Physical) ---")
    df.explain(True)
    
    spark.stop()
